# Notebook 2: Run Hardware Inference Benchmark

This notebook runs the fixed BFCL subset on one hardware backend.

It records:
- raw model output
- latency
- token counts
- tokens/sec
- GPU memory usage
- errors

The same notebook is used on each accelerator by changing the config file.

In [ ]:
!git clone https://github.com/joerosh/BFCL-hardware-efficiency-study.git
%cd BFCL-hardware-efficiency-study
!git pull
!pip install -r requirements.txt -q
!pip install accelerate bitsandbytes openai python-dotenv -q

Cloning into 'BFCL-hardware-efficiency-study'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 32 (delta 7), reused 31 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 47.02 KiB | 5.88 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/BFCL-hardware-efficiency-study


In [45]:
import json
import os
import time
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from openai import OpenAI

# XLA only imported if available — never crashes on GPU environments
try:
    import torch_xla.core.xla_model as xm
    XLA_AVAILABLE = True
except ImportError:
    xm = None
    XLA_AVAILABLE = False

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("XLA available:", XLA_AVAILABLE)

PyTorch version: 2.11.0
CUDA available: False
XLA available: False


In [ ]:
# Only run this cell if you are on a TPU runtime
# Skip entirely for GPU and API runs

if XLA_AVAILABLE:
    os.environ["PJRT_DEVICE"] = "TPU"
    device = xm.xla_device()
    print("XLA device:", device)
    try:
        print("XLA world size:", xm.xrt_world_size())
    except AttributeError:
        print("XLA world size unavailable in this torch_xla version.")
else:
    print("XLA not available — skipping TPU check. This is expected for GPU/API runs.")

In [47]:
# Paths — works whether running from notebooks/ or repo root
DATA_DIR    = Path("../data")
CONFIG_DIR  = Path("../configs")
RESULTS_DIR = Path("../results/raw")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TASK_PATH = DATA_DIR / "bfcl_subset.json"

# ── Set this to the config for your current run ──────────────────────────────
# CONFIG_PATH = CONFIG_DIR / "t4_config.json"             # T4 Qwen2.5-7B (done)
# CONFIG_PATH = CONFIG_DIR / "t4_llama_config.json"       # T4 Llama-3.1-8B
# CONFIG_PATH = CONFIG_DIR / "groq_llama_config.json"     # Groq LPU (done)
# CONFIG_PATH = CONFIG_DIR / "tpu_v3_llama_config.json"   # TPU v3 Kaggle
# CONFIG_PATH = CONFIG_DIR / "rtx6000_llama_config.json"    # RTX 6000 Ada school server
CONFIG_PATH = CONFIG_DIR / "cerebras_llama_config.json" # Cerebras WSE API
# ─────────────────────────────────────────────────────────────────────────────

def load_config(path):
    with open(path, "r", encoding="utf-8") as f:
        config = json.load(f)
    if "inherits" in config:
        parent_path = CONFIG_DIR / config["inherits"]
        with open(parent_path, "r", encoding="utf-8") as f:
            parent = json.load(f)
        parent.update({k: v for k, v in config.items() if k != "inherits"})
        return parent
    return config

config = load_config(CONFIG_PATH)
BACKEND    = config.get("backend", "transformers_gpu")
model_name = config["model_name"]

print("Backend:", BACKEND)
print("Hardware:", config["hardware"])
print("Model:", model_name)
print("Max new tokens:", config["max_new_tokens"])

Backend: cerebras_api
Hardware: Cerebras_WSE
Model: llama3.1-8b
Max new tokens: 512


In [48]:
with open(TASK_PATH, "r", encoding="utf-8") as f:
    tasks = json.load(f)

print(f"Loaded {len(tasks)} BFCL tasks.")
print("Categories:", pd.Series([t["category"] for t in tasks]).value_counts().to_dict())

Loaded 100 BFCL tasks.
Categories: {'simple': 25, 'multiple': 25, 'parallel': 25, 'parallel_multiple': 25}


## Hardware Check

In [49]:
if BACKEND == "transformers_gpu":
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("CUDA capability:", torch.cuda.get_device_capability(0))
        print("Total VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
    else:
        print("WARNING: No CUDA GPU detected.")

elif BACKEND == "transformers_xla":
    if not XLA_AVAILABLE:
        raise ImportError("torch_xla not available. Switch to a TPU runtime.")
    print("XLA device:", xm.xla_device())

else:
    print(f"API backend — no local hardware check needed: {BACKEND}")

API backend — no local hardware check needed: cerebras_api


## Load Model and Tokenizer

In [50]:
from dotenv import load_dotenv
load_dotenv()

tokenizer     = None
model         = None
groq_client   = None
xla_device    = None
together_client = None
cerebras_client = None

if BACKEND == "transformers_gpu":
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True,
        token=os.environ.get("HF_TOKEN")
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model_kwargs = {
        "torch_dtype": dtype,
        "device_map": "auto",
        "trust_remote_code": True,
        "token": os.environ.get("HF_TOKEN")
    }
    if config.get("load_in_8bit", False):
        model_kwargs["load_in_8bit"] = True
        model_kwargs.pop("torch_dtype", None)

    model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    model.eval()
    print("Loaded GPU model:", model_name)
    print("Device:", next(model.parameters()).device)
    print("Dtype:", next(model.parameters()).dtype)

elif BACKEND == "groq_prompt_api":
    groq_api_key = os.environ.get("GROQ_API_KEY")
    if not groq_api_key:
        raise ValueError("GROQ_API_KEY not set.")
    groq_client = OpenAI(
        base_url="https://api.groq.com/openai/v1",
        api_key=groq_api_key
    )
    print("Groq client configured:", model_name)

elif BACKEND == "together_api":
    together_api_key = os.environ.get("TOGETHER_API_KEY")
    if not together_api_key:
        raise ValueError("TOGETHER_API_KEY not set.")
    together_client = OpenAI(
        base_url="https://api.together.xyz/v1",
        api_key=together_api_key
    )
    print("Together AI client configured:", model_name)

elif BACKEND == "cerebras_api":
    try:
        from cerebras.cloud.sdk import Cerebras
    except ImportError:
        raise ImportError("Run: pip install cerebras-cloud-sdk")
    cerebras_api_key = os.environ.get("CEREBRAS_API_KEY")
    if not cerebras_api_key:
        raise ValueError("CEREBRAS_API_KEY not set.")
    cerebras_client = Cerebras(api_key=cerebras_api_key)
    print("Cerebras client configured:", model_name)

elif BACKEND == "transformers_xla":
    if not XLA_AVAILABLE:
        raise ImportError("torch_xla not available.")
    xla_device = xm.xla_device()
    tokenizer = AutoTokenizer.from_pretrained(
        model_name, trust_remote_code=True,
        token=os.environ.get("HF_TOKEN")
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.bfloat16,
        trust_remote_code=True,
        token=os.environ.get("HF_TOKEN")
    )
    model = model.to(xla_device)
    model.eval()
    print("Loaded XLA model:", model_name)

else:
    raise ValueError(f"Unsupported backend: {BACKEND}")

Cerebras client configured: llama3.1-8b


# Warmup

In [51]:
if BACKEND == "transformers_gpu":
    print("GPU warmup...")
    warmup_inputs = tokenizer("Hello", return_tensors="pt").to(model.device)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    with torch.no_grad():
        _ = model.generate(**warmup_inputs, max_new_tokens=5,
                           do_sample=False, pad_token_id=tokenizer.eos_token_id)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    print("GPU warmup complete.")

elif BACKEND == "groq_prompt_api":
    print("Groq warmup...")
    _ = groq_client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": "Say hello."}],
        max_tokens=5, temperature=0
    )
    print("Groq warmup complete.")

elif BACKEND == "together_api":
    print("Together AI warmup...")
    _ = together_client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": "Say hello."}],
        max_tokens=5, temperature=0
    )
    print("Together AI warmup complete.")

elif BACKEND == "cerebras_api":
    print("Cerebras warmup...")
    _ = cerebras_client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": "Say hello."}],
        max_completion_tokens=5
    )
    print("Cerebras warmup complete.")

elif BACKEND == "transformers_xla":
    print("XLA warmup — may take several minutes...")
    warmup_inputs = tokenizer("Hello", return_tensors="pt").to(xla_device)
    with torch.no_grad():
        _ = model.generate(**warmup_inputs, max_new_tokens=5,
                           do_sample=False, pad_token_id=tokenizer.eos_token_id)
    xm.mark_step()
    print("XLA warmup complete.")

else:
    print(f"No warmup for backend: {BACKEND}")

Cerebras warmup...
Cerebras warmup complete.


## Prompt Formatting

BFCL prompts are stored in chat-message format. Each task also includes function/tool schemas. We use the model tokenizer's chat template where possible.

In [52]:
def normalize_messages(prompt):
    if isinstance(prompt, list) and len(prompt) > 0:
        if isinstance(prompt[0], list):
            return prompt[0]
        if isinstance(prompt[0], dict):
            return prompt
    return [{"role": "user", "content": str(prompt)}]


def build_tool_prompt(task):
    messages  = normalize_messages(task["prompt"])
    user_content = messages[-1]["content"]
    tools = task["functions_or_tools"]
    system_content = (
        "You are a function-calling assistant.\n"
        "You must choose the correct function or functions from the available tools.\n"
        "Respond ONLY with tool call blocks in this exact format:\n"
        "<tool_call>\n"
        "{\"name\": \"function_name\", \"arguments\": {\"arg\": \"value\"}}\n"
        "</tool_call>\n\n"
        "If multiple calls are needed, output multiple <tool_call> blocks.\n\n"
        f"Available tools:\n{json.dumps(tools, indent=2)}"
    )
    return [
        {"role": "system", "content": system_content},
        {"role": "user",   "content": user_content}
    ]


def format_prompt(task):
    # For all GPU/TPU backends: always use build_tool_prompt
    # apply_chat_template with tools= silently drops schemas for Llama
    if BACKEND in ["transformers_gpu", "transformers_xla"]:
        prompt_messages = build_tool_prompt(task)
        return tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True
        )
    # All API backends return message list directly
    elif BACKEND in ["groq_prompt_api", "together_api", "cerebras_api"]:
        return build_tool_prompt(task)
    else:
        raise ValueError(f"Unsupported backend: {BACKEND}")


# Verify prompt format
sample = format_prompt(tasks[0])
if BACKEND in ["transformers_gpu", "transformers_xla"]:
    print("Prompt length:", len(sample))
    print("Contains tool_call:", "tool_call" in sample)
    print("First 500 chars:\n", sample[:500])
else:
    print("Messages:", len(sample))
    print("System prompt length:", len(sample[0]["content"]))
    print("Contains tool_call:", "tool_call" in sample[0]["content"])

Messages: 2
System prompt length: 1060
Contains tool_call: True


## Inference Function

This function runs one task, measures latency, counts tokens, and records GPU memory.

In [53]:
def run_one_task(task, run_index=0):
    result = {
        "task_id": task["task_id"],
        "category": task["category"],
        "hardware": config["hardware"],
        "model": config["model_name"],
        "backend": config.get("backend", "transformers_gpu"),
        "run_index": run_index,
        "raw_output": None,
        "latency_s": None,
        "input_tokens": None,
        "output_tokens": None,
        "tokens_per_second": None,
        "peak_memory_mb": None,
        "peak_memory_allocated_mb": None,
        "peak_memory_reserved_mb": None,
        "error": None,
        "groq_server_tokens_per_second": None,
        "groq_response_id": None,
        "timestamp": datetime.now(timezone.utc).isoformat()
    }

    try:
        if BACKEND == "transformers_gpu":
            prompt_text = format_prompt(task)
            inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
            input_tokens = inputs["input_ids"].shape[-1]

            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats()
                torch.cuda.synchronize()

            start = time.perf_counter()

            with torch.no_grad():
                output_ids = model.generate(
                    **inputs,
                    max_new_tokens=config["max_new_tokens"],
                    do_sample=False,
                    temperature=None,
                    pad_token_id=tokenizer.eos_token_id
                )

            if torch.cuda.is_available():
                torch.cuda.synchronize()

            latency_s = time.perf_counter() - start

            generated_ids = output_ids[0][input_tokens:]
            output_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

            output_tokens = generated_ids.shape[-1]
            tokens_per_second = output_tokens / latency_s if latency_s > 0 else None

            peak_memory_mb = None
            peak_memory_allocated_mb = None
            peak_memory_reserved_mb = None

            if torch.cuda.is_available():
                peak_memory_allocated_mb = torch.cuda.max_memory_allocated() / 1024**2
                peak_memory_reserved_mb = torch.cuda.max_memory_reserved() / 1024**2
                peak_memory_mb = peak_memory_allocated_mb

            result.update({
                "raw_output": output_text,
                "latency_s": latency_s,
                "input_tokens": int(input_tokens),
                "output_tokens": int(output_tokens),
                "tokens_per_second": tokens_per_second,
                "peak_memory_mb": peak_memory_mb,
                "peak_memory_allocated_mb": peak_memory_allocated_mb,
                "peak_memory_reserved_mb": peak_memory_reserved_mb
            })

        elif BACKEND == "groq_prompt_api":
            messages = format_prompt(task)

            start = time.perf_counter()

            response = groq_client.chat.completions.create(
                model=config["model_name"],
                messages=messages,
                temperature=config["temperature"],
                max_tokens=config["max_new_tokens"]
            )

            groq_server_tokens_per_second = None

            try:
                if hasattr(response, "x_groq") and response.x_groq:
                    groq_info = response.x_groq

                    if hasattr(groq_info, "usage") and groq_info.usage:
                        groq_usage = groq_info.usage

                        if hasattr(groq_usage, "completion_tokens_per_second"):
                            groq_server_tokens_per_second = groq_usage.completion_tokens_per_second
            except Exception:
                groq_server_tokens_per_second = None

            latency_s = time.perf_counter() - start

            output_text = response.choices[0].message.content or ""

            usage = getattr(response, "usage", None)
            input_tokens = getattr(usage, "prompt_tokens", None) if usage else None
            output_tokens = getattr(usage, "completion_tokens", None) if usage else None
            tokens_per_second = output_tokens / latency_s if output_tokens and latency_s > 0 else None

            result.update({
                "raw_output": output_text,
                "latency_s": latency_s,
                "input_tokens": input_tokens,
                "output_tokens": output_tokens,
                "tokens_per_second": tokens_per_second,
                "groq_server_tokens_per_second": groq_server_tokens_per_second,
                "groq_response_id": getattr(response, "id", None)
            })

        elif BACKEND == "transformers_xla":
            prompt_text = format_prompt(task)

            if config.get("pad_to_max_length", False):
                unpadded_inputs = tokenizer(prompt_text, return_tensors="pt")
                real_input_tokens = int(unpadded_inputs["input_ids"].shape[-1])

                inputs = tokenizer(
                    prompt_text,
                    return_tensors="pt",
                    padding="max_length",
                    max_length=config.get("max_input_length", 1024),
                    truncation=True
                ).to(xla_device)
            else:
                inputs = tokenizer(prompt_text, return_tensors="pt").to(xla_device)

            input_tokens = int(inputs["input_ids"].shape[-1])

            xm.mark_step()
            start = time.perf_counter()

            with torch.no_grad():
                output_ids = model.generate(
                    **inputs,
                    max_new_tokens=config["max_new_tokens"],
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id
                )

            xm.mark_step()
            latency_s = time.perf_counter() - start

            generated_ids = output_ids[0][input_tokens:]
            output_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

            output_tokens = int(generated_ids.shape[-1])
            tokens_per_second = output_tokens / latency_s if latency_s > 0 else None

            peak_memory_mb = None
            try:
                mem_info = xm.get_memory_info(xla_device)
                if isinstance(mem_info, dict):
                    peak_memory_mb = mem_info.get("bytes_used", None)
                    if peak_memory_mb is not None:
                        peak_memory_mb = peak_memory_mb / 1024**2
            except Exception:
                peak_memory_mb = None

            result.update({
                "raw_output": output_text,
                "latency_s": latency_s,
                "input_tokens": input_tokens,
                "output_tokens": output_tokens,
                "tokens_per_second": tokens_per_second,
                "peak_memory_mb": peak_memory_mb,
                "peak_memory_allocated_mb": peak_memory_mb,
                "peak_memory_reserved_mb": None, 
                "real_input_tokens": real_input_tokens,
            })

        elif BACKEND == "together_api":
            messages = format_prompt(task)

            start = time.perf_counter()

            response = together_client.chat.completions.create(
                model=config["model_name"],
                messages=messages,
                temperature=config["temperature"],
                max_tokens=config["max_new_tokens"]
            )

            latency_s = time.perf_counter() - start

            output_text = response.choices[0].message.content or ""

            usage = getattr(response, "usage", None)
            input_tokens = getattr(usage, "prompt_tokens", None) if usage else None
            output_tokens = getattr(usage, "completion_tokens", None) if usage else None
            tokens_per_second = output_tokens / latency_s if output_tokens and latency_s > 0 else None

            result.update({
                "raw_output": output_text,
                "latency_s": latency_s,
                "input_tokens": input_tokens,
                "output_tokens": output_tokens,
                "tokens_per_second": tokens_per_second,
            })
            
        elif BACKEND == "cerebras_api":
            messages = format_prompt(task)

            start = time.perf_counter()

            response = cerebras_client.chat.completions.create(
                model=model_name,
                messages=messages,
                max_completion_tokens=config["max_new_tokens"],
                temperature=config.get("temperature", 0)
            )

            latency_s = time.perf_counter() - start

            output_text = response.choices[0].message.content or ""

            usage = getattr(response, "usage", None)
            input_tokens  = getattr(usage, "prompt_tokens", None)     if usage else None
            output_tokens = getattr(usage, "completion_tokens", None) if usage else None
            tokens_per_second = (output_tokens / latency_s
                                 if output_tokens and latency_s > 0 else None)

            result.update({
                "raw_output":       output_text,
                "latency_s":        latency_s,
                "input_tokens":     input_tokens,
                "output_tokens":    output_tokens,
                "tokens_per_second": tokens_per_second,
            })

        else:
            raise ValueError(f"Unsupported backend: {BACKEND}")

    except Exception as e:
        result["error"] = str(e)

    return result

## Test Run on Two Tasks

Before running the full benchmark, test a small subset.

In [54]:
test_results = []

for task in tasks[:2]:
    result = run_one_task(task, run_index=0)
    test_results.append(result)
    print("\nTASK:", result["task_id"], result["category"])
    print("ERROR:", result["error"])
    print("LATENCY:", result["latency_s"])
    print("OUTPUT:\n", result["raw_output"][:1000] if result["raw_output"] else None)

pd.DataFrame(test_results)


TASK: simple_361 simple
ERROR: None
LATENCY: 0.32033133308868855
OUTPUT:
 <tool_call>
{"name": "restaurant_finder", "arguments": {"city": "New York", "cuisine": "Italian", "diet": "Gluten-free"}}
</tool_call>

TASK: simple_113 simple
ERROR: None
LATENCY: 0.18447150010615587
OUTPUT:
 <tool_call>
{"name": "probability.dice_roll", "arguments": {"desired_number": 6, "number_of_rolls": 2, "die_sides": 6}}
</tool_call>


,task_id,category,hardware,model,backend,run_index,raw_output,latency_s,input_tokens,output_tokens,tokens_per_second,peak_memory_mb,peak_memory_allocated_mb,peak_memory_reserved_mb,error,groq_server_tokens_per_second,groq_response_id,timestamp
0,simple_361,simple,Cerebras_WSE,llama3.1-8b,cerebras_api,0,"<tool_call>\n{""name"": ""restaurant_finder"", ""ar...",0.320331,296,41,127.992475,None,None,None,None,None,None,2026-04-30T21:50:05.793196+00:00
1,simple_113,simple,Cerebras_WSE,llama3.1-8b,cerebras_api,0,"<tool_call>\n{""name"": ""probability.dice_roll"",...",0.184472,308,45,243.940121,None,None,None,None,None,None,2026-04-30T21:50:06.113862+00:00


## Full Benchmark Run

This writes results incrementally as JSONL so partial runs are recoverable.

In [55]:
hardware = config["hardware"]
output_path = RESULTS_DIR / f"{hardware}_results.jsonl"

print("Output path:", output_path)

Output path: ../results/raw/Cerebras_WSE_results.jsonl


In [56]:
if BACKEND == "groq_prompt_api":
    print("Warming up Groq API...")

    _ = groq_client.chat.completions.create(
        model=config["model_name"],
        messages=[{"role": "user", "content": "Say hello."}],
        max_tokens=5,
        temperature=0
    )

    print("Groq warmup complete.")
else:
    print(f"No API warmup needed for backend: {BACKEND}")


RUN_FULL_BENCHMARK = True  # Change to True when ready

if RUN_FULL_BENCHMARK:
    with open(output_path, "a", encoding="utf-8") as f:
        for i, task in enumerate(tasks):
            print(f"[{i+1}/{len(tasks)}] Running {task['task_id']} ({task['category']})")

            result = run_one_task(task, run_index=0)

            f.write(json.dumps(result) + "\n")
            f.flush()

            print("  error:", result["error"])
            print("  latency:", result["latency_s"])
            print("  toks/sec:", result["tokens_per_second"])

            if BACKEND in ["groq_prompt_api", "together_api", "cerebras_api"]:
                time.sleep(1)  # 1 second between requests = ~60 req/min max

    print("Benchmark complete.")
else:
    print("RUN_FULL_BENCHMARK is False. Set it to True to run all tasks.")

No API warmup needed for backend: cerebras_api
[1/100] Running simple_361 (simple)
  error: None
  latency: 0.30102204205468297
  toks/sec: 136.20265054394932
[2/100] Running simple_113 (simple)
  error: None
  latency: 0.15705475001595914
  toks/sec: 286.52428529176814
[3/100] Running simple_135 (simple)
  error: None
  latency: 0.366244541015476
  toks/sec: 117.40789331842355
[4/100] Running simple_1 (simple)
  error: None
  latency: 0.24853883299510926
  toks/sec: 104.61141901519923
[5/100] Running simple_123 (simple)
  error: None
  latency: 0.2659605420194566
  toks/sec: 304.55645557405416
[6/100] Running simple_36 (simple)
  error: None
  latency: 0.2643609589431435
  toks/sec: 170.22180650236717
[7/100] Running simple_353 (simple)
  error: None
  latency: 0.21510324999690056
  toks/sec: 167.3614880319973
[8/100] Running simple_137 (simple)
  error: None
  latency: 0.24756454105954617
  toks/sec: 181.7707810957315
[9/100] Running simple_200 (simple)
  error: None
  latency: 0.168

## Load Saved Results

In [57]:
def load_jsonl(path):
    rows = []
    if not path.exists():
        return rows

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))

    return rows

saved_rows = load_jsonl(output_path)
df_results = pd.DataFrame(saved_rows)

print("Saved rows:", len(df_results))
df_results.head()

Saved rows: 100


,task_id,category,hardware,model,backend,run_index,raw_output,latency_s,input_tokens,output_tokens,tokens_per_second,peak_memory_mb,peak_memory_allocated_mb,peak_memory_reserved_mb,error,groq_server_tokens_per_second,groq_response_id,timestamp
0,simple_361,simple,Cerebras_WSE,llama3.1-8b,cerebras_api,0,"<tool_call>\n{""name"": ""restaurant_finder"", ""ar...",0.301022,296,41,136.202651,None,None,None,None,None,None,2026-04-30T21:50:15.845799+00:00
1,simple_113,simple,Cerebras_WSE,llama3.1-8b,cerebras_api,0,"<tool_call>\n{""name"": ""probability.dice_roll"",...",0.157055,308,45,286.524285,None,None,None,None,None,None,2026-04-30T21:50:17.152366+00:00
2,simple_135,simple,Cerebras_WSE,llama3.1-8b,cerebras_api,0,"<tool_call>\n{""name"": ""calculate_return_on_inv...",0.366245,306,43,117.407893,None,None,None,None,None,None,2026-04-30T21:50:18.316729+00:00
3,simple_1,simple,Cerebras_WSE,llama3.1-8b,cerebras_api,0,"<tool_call>\n{""name"": ""math.factorial"", ""argum...",0.248539,207,26,104.611419,None,None,None,None,None,None,2026-04-30T21:50:19.696762+00:00
4,simple_123,simple,Cerebras_WSE,llama3.1-8b,cerebras_api,0,"<tool_call>\n{""name"": ""hypothesis_testing.two_...",0.265961,389,81,304.556456,None,None,None,None,None,None,2026-04-30T21:50:20.953028+00:00


In [58]:
if len(df_results) > 0:
    display(df_results.groupby("category")[["latency_s", "tokens_per_second", "peak_memory_mb"]].mean())
    display(df_results[["latency_s", "tokens_per_second", "input_tokens", "output_tokens", "peak_memory_mb"]].describe())
else:
    print("No saved rows yet.")

,latency_s,tokens_per_second,peak_memory_mb
category,,,
multiple,0.262730,171.969609,NaN
parallel,2.685287,394.363298,NaN
parallel_multiple,0.312461,387.435193,NaN
simple,0.253864,185.698613,NaN


,latency_s,tokens_per_second,input_tokens,output_tokens
count,100.000000,100.000000,100.000000,100.000000
mean,0.878585,284.866678,447.780000,83.550000
std,5.918782,151.476822,165.963156,50.045711
min,0.157055,1.429288,207.000000,26.000000
25%,0.236854,175.787144,326.000000,43.000000
50%,0.276868,239.957852,399.000000,64.500000
75%,0.329240,371.682238,561.000000,118.500000
max,59.470158,745.315448,994.000000,212.000000


In [ ]:
# from google.colab import files
# files.download("results/raw/T4_results.jsonl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>